In [ ]:
!nvidia-smi

Wed Aug 26 08:51:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip -q install "transformers==4.46.*" "accelerate==1.1.*"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 118.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
%%writefile generate_probe.py
"""Device probe: the SAME code, run in two places, is the proof."""
from __future__ import annotations

import json
import os
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = os.environ.get("MODEL_ID", "Qwen/Qwen2.5-1.5B-Instruct")

if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.float16
    device_name = torch.cuda.get_device_name(0)
else:
    device = "cpu"
    dtype = torch.float32
    device_name = "cpu"

print(f"device: {device} ({device_name}), dtype: {dtype}")

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype)
model.to(device)
model.eval()

msgs = [{"role": "user", "content": "Explain what a GPU does, in three sentences."}]
ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(device)

with torch.no_grad():
    model.generate(ids, max_new_tokens=8)

if device == "cuda":
    torch.cuda.synchronize()
t0 = time.time()
with torch.no_grad():
    out = model.generate(ids, max_new_tokens=128, do_sample=False)
if device == "cuda":
    torch.cuda.synchronize()
dt = time.time() - t0

generated = out.shape[1] - ids.shape[1]
tokens_per_s = generated / dt

evidence = {
    "cuda": device == "cuda",
    "device_name": device_name,
    "tokens_per_s": round(tokens_per_s, 1),
    "model": MODEL_ID,
}

with open("gpu_evidence.json", "w") as f:
    json.dump(evidence, f, indent=2)

print(json.dumps(evidence, indent=2))
print("wrote gpu_evidence.json")

Writing generate_probe.py


In [ ]:
!python generate_probe.py

device: cuda (Tesla T4), dtype: torch.float16
tokenizer_config.json: 7.30kB [00:00, 33.6MB/s]
vocab.json: 2.78MB [00:00, 75.9MB/s]
merges.txt: 1.67MB [00:00, 146MB/s]
tokenizer.json: 7.03MB [00:00, 167MB/s]
config.json: 100% 660/660 [00:00<00:00, 5.60MB/s]
2026-08-26 09:02:37.160747: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
model.safetensors: 100% 3.09G/3.09G [00:29<00:00, 104MB/s]
generation_config.json: 100% 242/242 [00:00<00:00, 2.61MB/s]
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
/usr/local/lib/python3.13/dist-packages/transformers/generation/configurat

In [ ]:
from google.colab import files
files.download('gpu_evidence.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>